# 2. Training metrics
This notebook reads saved training artifacts. The current executable model records **training reconstruction loss**, not validation loss or ELBO. A decreasing curve shows optimization progress, but does not prove generalization or fashion compatibility.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'config.yml').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.utils.setup_utilities import load_config
config = load_config(PROJECT_ROOT / 'config.yml')
training = config['training']
print('Project:', PROJECT_ROOT)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

metrics_path = Path(training['metrics_path'])
if not metrics_path.is_file():
    raise FileNotFoundError('Training metrics are missing. Run: python -m scripts.run_pipeline --from-stage train --to-stage train')
metrics = pd.read_csv(metrics_path)
display(metrics.tail())
summary = pd.Series({
    'epochs completed': len(metrics),
    'first loss': metrics.reconstruction_loss.iloc[0],
    'last loss': metrics.reconstruction_loss.iloc[-1],
    'best loss': metrics.reconstruction_loss.min(),
    'best epoch': int(metrics.loc[metrics.reconstruction_loss.idxmin(), 'epoch']),
    'total training hours': metrics.epoch_seconds.sum() / 3600,
})
display(summary.to_frame('value'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(metrics.epoch, metrics.reconstruction_loss)
axes[0].set(title='Training reconstruction loss', xlabel='Epoch', ylabel='Mean binary cross-entropy')
axes[0].grid(alpha=0.25)
axes[1].plot(metrics.epoch, metrics.epoch_seconds)
axes[1].set(title='Time per epoch', xlabel='Epoch', ylabel='Seconds')
axes[1].grid(alpha=0.25)
plt.tight_layout()

The exported vectors are posterior means, so repeated retrieval is deterministic. The following distributions can reveal collapsed or unusually large embeddings, but they are diagnostic rather than accuracy metrics.

In [ ]:
import pickle
import numpy as np

with open(training['model_data_path'], 'rb') as stream:  # Only load pickle files you created and trust.
    model_data = pickle.load(stream)
embeddings = np.asarray(model_data['latent_space'])
norms = np.linalg.norm(embeddings, axis=1)
display(pd.Series({
    'outfit embeddings': len(embeddings),
    'latent dimensions': embeddings.shape[1],
    'mean vector norm': norms.mean(),
    'std vector norm': norms.std(),
    'exact zero values (%)': 100 * (embeddings == 0).mean(),
}, name='value').to_frame())
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(norms, bins=40)
axes[0].set(title='Embedding norm distribution', xlabel='L2 norm', ylabel='Outfits')
axes[1].hist(embeddings.ravel(), bins=60)
axes[1].set(title='Latent value distribution', xlabel='Value', ylabel='Count')
plt.tight_layout()

In [ ]:
from IPython.display import Image, display

previews = sorted(Path(training['epoch_images_path']).glob('image_at_epoch_*.png'))
if previews:
    print('Latest reconstruction preview:', previews[-1].name)
    display(Image(filename=str(previews[-1])))
else:
    print('No reconstruction preview exists yet.')